# Observe the ReAct cycle

You are on a practice security team at a company. This notebook prints simple status updates while the agent works.

![ReAct loop event stream](figures/react-loop-event-stream.svg)

The first model request asks for a tool. The tool result returns to the agent, and the second model request writes the answer.

## Step 1: Import the lesson components

This notebook does not write its own Python while loop. Calling reply_stream starts AgentScope's built-in ReAct cycle and gives the notebook each status update as it happens.

## How the cycle works

1. The agent sends the employee's question and its instructions to the model.
2. The model asks to use get_ip_details because it needs information from the practice list.
3. AgentScope runs the Python function and gives its result back to the model.
4. The model uses the result to write the answer.

One tool is enough to show the cycle: model request → tool → model request → answer. The setting max_iters=3 is a safety limit: it stops the agent if it keeps asking for tools instead of reaching an answer.

In [ ]:
# Counter records how many times each important ReAct event occurs.
from collections import Counter
import os

# load_dotenv reads MODEL and OLLAMA_BASE_URL from the local .env file.
from dotenv import load_dotenv

# Agent runs the ReAct cycle; ReActConfig supplies its safety limits.
from agentscope.agent import Agent, ReActConfig
from agentscope.credential import OpenAICredential

# Msg is the question sent to the agent. TextBlock holds text within a message.
from agentscope.message import Msg, TextBlock
from agentscope.model import OpenAIChatModel

# FunctionTool makes a Python function available to the model; Toolkit holds that tool.
from agentscope.tool import FunctionTool, Toolkit

## Step 2: Define the practice tool and event translator

The next cell creates the fixed IP-details tool and a helper that turns AgentScope event objects into short, readable status messages.

In [ ]:
def get_ip_details(ip_address: str) -> dict[str, str]:
    """Get details from the fixed practice list.

    Inputs:
        ip_address: An internet address, such as 192.0.2.44.
    Output:
        A matching record or a no-record result.
    Process:
        1. Search the fixed list using the input address.
        2. Return saved details when found.
        3. Otherwise return no record.
    """
    # This fixed dictionary is intentionally small: the lesson observes the
    # agent's process, not a real IP-address lookup service.
    records = {
        "192.0.2.44": {"local_result": "suspicious", "details": "A person should look into this address in the practice scenario."},
        "198.51.100.10": {"local_result": "known benign", "details": "Expected activity in the practice scenario."},
    }
    # .get returns the matching record, or the second dictionary when the
    # address is absent. This makes the tool's no-record behavior explicit.
    return records.get(ip_address, {"local_result": "no record", "details": "The practice list has no details for this address."})


def describe_event(event) -> str:
    """Turn an AgentScope update into plain language.

    Inputs:
        event: One status update from the agent.
    Output:
        A short sentence explaining the update.
    Process:
        1. Read the update name.
        2. Match known names to simple sentences.
        3. Use a general sentence for other updates.
    """
    # AgentScope yields event objects of several classes. Their class name is
    # enough for this notebook to identify the stage of the ReAct cycle.
    name = event.__class__.__name__
    messages = {
        "ModelCallStartEvent": "Model request starts.",
        # getattr prevents a display error if an event has no tool name.
        "ToolCallStartEvent": f"Tool requested: {getattr(event, 'tool_call_name', 'unknown tool')}.",
        "ToolResultStartEvent": f"Tool starts: {getattr(event, 'tool_call_name', 'unknown tool')}.",
        "ToolResultEndEvent": "Tool result is ready.",
        "ReplyEndEvent": "Agent finished its answer.",
    }
    # Other AgentScope updates can still be useful, so show their real name.
    return messages.get(name, f"Update: {name}")

# FunctionTool exposes the normal Python function to the agent. Marking it
# read-only documents that it only reads this practice data.
ip_details_tool = FunctionTool(get_ip_details, is_read_only=True)

# The agent can call only tools placed in its Toolkit.
toolkit = Toolkit(tools=[ip_details_tool])

## Step 3: Configure the tool-using agent

| Function | Input | Output |
| --- | --- | --- |
| get_ip_details | 192.0.2.44 | A record marked suspicious. |
| get_ip_details | 198.51.100.23 | A no-record result. |
| describe_event | a tool-start update | Tool starts: get_ip_details. |

In [ ]:
# Read connection settings without placing machine-specific values in the notebook.
load_dotenv()
model_name = os.getenv("MODEL")
base_url = os.getenv("OLLAMA_BASE_URL")

# Stop early with a clear fix if .env is missing or incomplete.
if not model_name or not base_url:
    raise RuntimeError("Set MODEL and OLLAMA_BASE_URL in .env before running this notebook.")

# This local OpenAI-compatible endpoint is provided by Ollama. stream=False
# applies to each model call; reply_stream below still yields AgentScope events.
model = OpenAIChatModel(
    credential=OpenAICredential(api_key="ollama", base_url=base_url),
    model=model_name,
    stream=False,
    # Deterministic, short answers make the event sequence easier to compare.
    parameters=OpenAIChatModel.Parameters(temperature=0, max_tokens=180),
)

# The prompt requires tool use, so a typical run reveals the full ReAct cycle.
# max_iters is a guardrail: AgentScope stops after three iterations if the
# model repeatedly asks for tools instead of producing an answer.
agent = Agent(
    name="practice_assistant",
    system_prompt="For every internet-address question, call get_ip_details before answering. Use only the tool result.",
    model=model,
    toolkit=toolkit,
    react_config=ReActConfig(max_iters=3),
)

## Step 4: Run the agent and watch the event stream

The next cell sends a practice question, prints each event as it arrives, and counts the major events so you can compare the completed ReAct cycle with the expected sequence.

In [ ]:
# A Msg is the structured user request AgentScope sends to the agent.
question = Msg(
    name="analyst",
    role="user",
    content=[TextBlock(text="What details does the practice list have about 192.0.2.44?")],
)

# Keep a count separate from the printed trace. This makes it easy to check
# whether the expected cycle occurred even when extra events are emitted.
important_events = Counter()

# Translate AgentScope class names into labels suitable for a beginner-facing
# summary. Events not in this dictionary are printed but not counted.
labels = {
    "ModelCallStartEvent": "Model request starts",
    "ToolCallStartEvent": "Tool requested",
    "ToolResultStartEvent": "Tool starts",
    "ToolResultEndEvent": "Tool result is ready",
    "ReplyEndEvent": "Agent finished its answer",
}

# reply_stream runs AgentScope's built-in ReAct cycle. As each event arrives,
# the async loop pauses until the next event is available. yield_final_msg=True
# also gives us the final Msg, not just the intermediate status events.
async for item in agent.reply_stream(question, yield_final_msg=True):
    # The final result is a Msg. Everything else yielded here is an event that
    # lets us observe work normally hidden inside agent.reply(...).
    if isinstance(item, Msg):
        print("FINAL ANSWER:")
        # A message can contain several block types. Print only text blocks.
        print("".join(block.text for block in item.content if isinstance(block, TextBlock)))
    else:
        event_name = item.__class__.__name__
        # Count the selected events using their readable labels. Counter returns
        # zero for labels that never appear, which helps reveal a missing step.
        if event_name in labels:
            important_events[labels[event_name]] += 1
        # Print every event, including uncounted ones, for full visibility.
        print(describe_event(item))

print("\nEVENT SUMMARY:")
# Print labels in a fixed order so separate runs are easy to compare.
for label in labels.values():
    print(f"{label}: {important_events[label]}")